In [1]:
# If you're on Colab, turn on GPU: Runtime > Change runtime type > T4 GPU (or any GPU)

!pip -q install --upgrade transformers datasets accelerate scikit-learn evaluate matplotlib

import os, sys, random, numpy as np, torch

print("Python:", sys.version)
print("Transformers / Datasets installed ✅")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Transformers / Datasets installed ✅
Torch: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4


In [2]:
import re
from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import confusion_matrix, classification_report

import datasets
from datasets import load_dataset, Dataset, DatasetDict

import torch
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [4]:
# Cell 3 — Load Dataset without Hugging Face (robust, 3 fallbacks)

import pandas as pd
import re
from pathlib import Path

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize common spam dataset schemas to columns: text (str) and label (0/1)."""
    # Lowercase column names
    df.columns = [c.strip().lower() for c in df.columns]

    # Common layouts:
    # 1) UCI/Kaggle: v1 (ham/spam), v2 (text)
    if set(df.columns) >= {"v1", "v2"}:
        df = df.rename(columns={"v1": "label", "v2": "text"})

    # 2) Markham pycon TSV: label, message
    elif set(df.columns) >= {"label", "message"}:
        df = df.rename(columns={"message": "text"})

    # 3) Generic: label, text
    elif set(df.columns) >= {"label", "text"}:
        pass

    else:
        raise ValueError(
            f"Could not infer (label, text) from columns {list(df.columns)}. "
            "Expected something like (v1,v2) or (label,message) or (label,text)."
        )

    # Map labels to 0/1 if needed
    if df["label"].dtype == object:
        mapping = {"ham": 0, "spam": 1, "Ham": 0, "Spam": 1, "HAM": 0, "SPAM": 1}
        if df["label"].iloc[0] in mapping or df["label"].str.lower().isin(["ham", "spam"]).any():
            df["label"] = df["label"].str.lower().map({"ham": 0, "spam": 1}).astype(int)
        else:
            # If already numeric as strings like "0"/"1"
            df["label"] = df["label"].astype(int)

    # Keep only required cols, drop bad rows & duplicates
    df = df[["text", "label"]].dropna().drop_duplicates(subset=["text"]).reset_index(drop=True)

    # Ensure label is int and text is str
    df["label"] = df["label"].astype(int)
    df["text"] = df["text"].astype(str)
    return df

def load_spam_df() -> pd.DataFrame:
    """
    Strategy:
      A) Try a well-known public TSV mirror (label ru/ham, text column 'message')
      B) If that fails, prompt for manual CSV upload
      C) If user cancels upload, fall back to a tiny in-notebook sample so the pipeline runs
    """
    # A) Try public TSV (commonly used pycon tutorial mirror)
    url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
    try:
        df = pd.read_csv(url, sep="\t", header=0, names=["label", "message"])
        df = normalize_df(df)
        print("Loaded dataset from public TSV mirror ✅")
        return df
    except Exception as e:
        print("Public TSV load failed. Reason:", e)

    # B) Manual upload (CSV/TSV). Expect columns like (v1,v2) or (label,message) or (label,text)
    try:
        from google.colab import files
        print("Please upload your dataset CSV/TSV (e.g., spam.csv).")
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        # Try CSV first, then TSV
        try:
            df = pd.read_csv(fname, encoding="latin-1", on_bad_lines="skip")
        except Exception:
            df = pd.read_csv(fname, sep="\t", encoding="latin-1", on_bad_lines="skip")
        df = normalize_df(df)
        print("Loaded dataset from your upload ✅")
        return df
    except Exception as e:
        print("No file uploaded or parsing failed. Reason:", e)

    # C) Tiny built-in sample (so you can continue the notebook)
    print("Using a tiny built-in sample so you can proceed ✅")
    sample = [
        {"text": "Congratulations! You have won a free prize. Claim now!", "label": 1},
        {"text": "Are we meeting at 5 pm in the lab?", "label": 0},
        {"text": "URGENT! Your account is compromised. Verify at http://fake.example", "label": 1},
        {"text": "The submission is due tomorrow, please upload.", "label": 0},
        {"text": "You have been selected for a free gift card!", "label": 1},
        {"text": "Mom called, call her back when free.", "label": 0},
    ]
    return pd.DataFrame(sample)

# Load → convert to HF DatasetDict with a single 'full' split to match later cells
from datasets import Dataset, DatasetDict

df_full = load_spam_df()
ds_full = Dataset.from_pandas(df_full, preserve_index=False)
ds_all = DatasetDict({"full": ds_full})

print(ds_all)
print("Samples:", len(ds_all["full"]))
print(ds_all["full"][0])


Loaded dataset from public TSV mirror ✅
DatasetDict({
    full: Dataset({
        features: ['text', 'label'],
        num_rows: 5168
    })
})
Samples: 5168
{'text': 'Ok lar... Joking wif u oni...', 'label': 0}


In [5]:
def clean_text(t: str) -> str:
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"[^A-Za-z0-9\s']", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t.lower()

df_full = ds_all["full"].to_pandas()
df_full["text"] = df_full["text"].astype(str).apply(clean_text)
df_full = df_full[df_full["text"].str.len() > 0].reset_index(drop=True)

print(df_full.head(3))
print(df_full["label"].value_counts())

# Rebuild HF dataset
ds_full = Dataset.from_pandas(df_full, preserve_index=False)


                                                text  label
0                            ok lar joking wif u oni      0
1  free entry in 2 a wkly comp to win fa cup fina...      1
2        u dun say so early hor u c already then say      0
label
0    4513
1     653
Name: count, dtype: int64


In [6]:
# 80/10/10 split
ds_split = ds_full.train_test_split(test_size=0.2, seed=SEED)
ds_temp = ds_split["train"].train_test_split(test_size=0.125, seed=SEED)  # 0.125 of 0.8 = 0.1
ds = DatasetDict({
    "train": ds_temp["train"],
    "validation": ds_temp["test"],
    "test": ds_split["test"]
})
ds


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3615
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 517
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1034
    })
})

In [7]:
MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

MAX_LEN = 128

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,   # let data collator pad dynamically
        max_length=MAX_LEN
    )

tokenized = ds.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/3615 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3615
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 517
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1034
    })
})

In [8]:
num_labels = 2
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# Put model on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [9]:
metric_accuracy = evaluate.load("accuracy")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    results = {
        "accuracy": metric_accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": metric_precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": metric_recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": metric_f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }
    return results


In [11]:
from transformers import TrainingArguments, Trainer

batch_size = 16

training_args = TrainingArguments(
    output_dir="spam_bert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",        # ✅ renamed for newer versions
    save_strategy="epoch",        # still valid
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to=[],                 # ✅ changed from "none"
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()



/tmp/ipython-input-1002644857.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.057700,0.043303,0.990329,1.000000,0.930556,0.964029
2,0.025100,0.013640,0.996132,0.986111,0.986111,0.986111
3,0.001000,0.018520,0.994197,1.000000,0.958333,0.978723


TrainOutput(global_step=678, training_loss=0.047002258323198924, metrics={'train_runtime': 150.6833, 'train_samples_per_second': 71.972, 'train_steps_per_second': 4.5, 'total_flos': 285862967205720.0, 'train_loss': 0.047002258323198924, 'epoch': 3.0})

In [12]:
test_out = trainer.predict(tokenized["test"])
logits = test_out.predictions
y_pred = np.argmax(logits, axis=-1)
y_true = np.array(test_out.label_ids)

print("Test Metrics:", compute_metrics((logits, y_true)))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["ham(0)", "spam(1)"]))

cm = confusion_matrix(y_true, y_pred)
cm


Test Metrics: {'accuracy': 0.9903288201160542, 'precision': 0.9769230769230769, 'recall': 0.9477611940298507, 'f1': 0.9621212121212122}

Classification Report:
               precision    recall  f1-score   support

      ham(0)       0.99      1.00      0.99       900
     spam(1)       0.98      0.95      0.96       134

    accuracy                           0.99      1034
   macro avg       0.98      0.97      0.98      1034
weighted avg       0.99      0.99      0.99      1034



array([[897,   3],
       [  7, 127]])

In [13]:
examples = [
    "Congratulations! You won a FREE prize. Click here to claim now!",
    "Bro are we meeting at 5 pm near the lab?",
    "Important notice: Your account has been compromised. Verify immediately at http://secure-verify.example",
    "Final project submission is tomorrow. Please upload to Google Drive.",
]

model.eval()
inputs = tokenizer(examples, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
with torch.no_grad():
    outputs = model(**inputs)
preds = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

for txt, p in zip(examples, preds):
    pred_label = int(np.argmax(p))
    print(f"[{'SPAM' if pred_label==1 else 'HAM '}] {txt}\n    probs -> ham={p[0]:.3f}, spam={p[1]:.3f}\n")


[SPAM] Congratulations! You won a FREE prize. Click here to claim now!
    probs -> ham=0.015, spam=0.985

[HAM ] Bro are we meeting at 5 pm near the lab?
    probs -> ham=0.999, spam=0.001

[SPAM] Important notice: Your account has been compromised. Verify immediately at http://secure-verify.example
    probs -> ham=0.007, spam=0.993

[HAM ] Final project submission is tomorrow. Please upload to Google Drive.
    probs -> ham=0.803, spam=0.197



In [14]:
save_dir = "spam_bert_model"
os.makedirs(save_dir, exist_ok=True)
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Saved to: {os.path.abspath(save_dir)}")

# (Optional) Save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r spam_bert_model /content/drive/MyDrive/


Saved to: /content/spam_bert_model


In [15]:
from datetime import datetime
log = f"""
[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Run Summary:
- Model: {MODEL_NAME}
- Epochs: {training_args.num_train_epochs}
- Batch size: {batch_size}
- Max length: {MAX_LEN}
- Test metrics: {compute_metrics((logits, y_true))}
- Confusion matrix: {confusion_matrix(y_true, y_pred).tolist()}
"""
print(log)



[2025-11-06 07:44:20] Run Summary:
- Model: bert-base-uncased
- Epochs: 3
- Batch size: 16
- Max length: 128
- Test metrics: {'accuracy': 0.9903288201160542, 'precision': 0.9769230769230769, 'recall': 0.9477611940298507, 'f1': 0.9621212121212122}
- Confusion matrix: [[897, 3], [7, 127]]



In [17]:
from transformers import BertTokenizerFast, BertForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 128  # same as used in training

# load from your saved model folder
model_dir = "spam_bert_model"
tokenizer = BertTokenizerFast.from_pretrained(model_dir)
model = BertForSequenceClassification.from_pretrained(model_dir).to(device)
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [19]:
# --- Install & imports
!pip -q install gradio

import os, json, torch, numpy as np, gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- Config
MODEL_DIR = "spam_bert_model"   # <- change if you saved to a different folder
MAX_LEN   = 128                 # same as training
THRESHOLD = 0.5                 # tune if you adjusted it on validation
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load (robust)
assert os.path.isdir(MODEL_DIR), f"Model folder not found: {MODEL_DIR}"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

# --- Predictor (robust, with clear error messages)
def predict_spam(text: str):
    try:
        text = (text or "").strip()
        if not text:
            return "Enter a message", {"ham(0)": 1.0, "spam(1)": 0.0}

        inputs = tokenizer(
            [text],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        ).to(DEVICE)

        with torch.no_grad():
            out = model(**inputs)
            probs = torch.softmax(out.logits, dim=-1).cpu().numpy()[0].astype(float)

        p_ham, p_spam = float(probs[0]), float(probs[1])
        label = "SPAM" if p_spam >= THRESHOLD else "HAM"
        return label, {"ham(0)": p_ham, "spam(1)": p_spam}

    except Exception as e:
        # Return a friendly message instead of crashing the UI
        msg = f"Runtime error: {type(e).__name__}: {e}"
        # Show as HAM by default with zeros so component renders
        return msg, {"ham(0)": 0.0, "spam(1)": 0.0}

# --- Quick self-test (so you can see errors in the notebook output)
_test_label, _test_probs = predict_spam("Congratulations! You won a prize!")
print("Self-test:", _test_label, _test_probs)

# --- Gradio app (simple & stable)
demo = gr.Interface(
    fn=predict_spam,
    inputs=gr.Textbox(lines=3, label="Enter message"),
    outputs=[gr.Label(label="Prediction"), gr.Label(label="Probabilities")],
    title="Spam Detection (BERT)",
    description=f"Threshold = {THRESHOLD}. Type a message to classify as SPAM or HAM."
)

demo.launch(share=True)



Self-test: HAM {'ham(0)': 0.9964104294776917, 'spam(1)': 0.0035895940382033587}
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ac4b8279bf510e0a26.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
